In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/6cityyyyy/unified_air_quality.csv


In [5]:
import pandas as pd
import numpy as np
import os
import gc
from tqdm.notebook import tqdm # Use notebook tqdm

# --- 1. Configuration ---
# *** ADJUST THESE PATHS based on your Kaggle input folder structure ***
UNIFIED_DATA_PATH = "/kaggle/input/6cityyyyy/unified_air_quality.csv"
# STATIONS_INV_PATH = "/kaggle/input/your-stations-folder/stations_inventory.csv" # Might not be needed if state/city are in unified data
OUTPUT_DIR = "/kaggle/working/"

DATE_COL = 'datetime' # The column with parsed datetime objects in unified_air_quality.csv
CITY_COL = 'city'
STATE_COL = 'state' # Ensure this column exists in unified_air_quality.csv
STATION_COL = 'station_id' # Ensure this column exists if you want to keep station-level detail initially

# Pollutant columns in unified_air_quality.csv (use the cleaned names)
POLLUTANT_COLS = ['PM2_5', 'PM10', 'NO', 'NO2', 'SO2', 'CO', 'NH3'] # Add/remove based on your actual columns

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration set.")

# --- 2. Load Unified Data ---
print(f"Loading unified data from: {UNIFIED_DATA_PATH}")
try:
    # Load data, ensuring the datetime column is parsed correctly
    df_unified = pd.read_csv(UNIFIED_DATA_PATH, parse_dates=[DATE_COL], low_memory=False)
    print(f"Loaded unified data shape: {df_unified.shape}")
    print(f"Columns found: {df_unified.columns.tolist()}")

    # Basic check for essential columns
    essential_cols_load = [DATE_COL, CITY_COL, STATE_COL] + [p for p in POLLUTANT_COLS if p in df_unified.columns]
    missing_essential = [col for col in essential_cols_load if col not in df_unified.columns]
    if missing_essential:
         raise ValueError(f"Essential columns missing from loaded data: {missing_essential}")
    if not pd.api.types.is_datetime64_any_dtype(df_unified[DATE_COL]):
        print(f"Warning: Column '{DATE_COL}' was not parsed as datetime. Attempting conversion...")
        df_unified[DATE_COL] = pd.to_datetime(df_unified[DATE_COL], errors='coerce')
        if df_unified[DATE_COL].isnull().all():
             raise ValueError(f"Failed to convert '{DATE_COL}' to datetime.")
        df_unified = df_unified.dropna(subset=[DATE_COL])


except FileNotFoundError:
    print(f"ERROR: File not found at {UNIFIED_DATA_PATH}. Please check the path in Configuration.")
    # Stop execution if file not found
    raise
except Exception as e:
    print(f"ERROR loading or processing unified data: {e}")
    raise

# --- 3. Identify Top 6 Cities by Data Count ---
print("\nIdentifying top 6 cities based on data availability...")
city_counts = df_unified[CITY_COL].value_counts()

if len(city_counts) == 0:
    raise ValueError("No cities found in the dataset. Check the CITY_COL name and data.")
elif len(city_counts) < 6:
    print(f"Warning: Found only {len(city_counts)} cities. Selecting all.")
    top6_cities = city_counts.index.tolist()
else:
    top6_cities = city_counts.head(6).index.tolist()

print(f"\nTop 6 cities selected: {top6_cities}")
print("Data counts for selected cities:")
print(city_counts.loc[top6_cities])

# --- 4. Filter Unified Data for Top 6 Cities ---
print(f"\nFiltering unified data for {len(top6_cities)} cities...")
df_filtered_hourly = df_unified[df_unified[CITY_COL].isin(top6_cities)].copy()
print(f"Shape after filtering (hourly): {df_filtered_hourly.shape}")

# Optional: Save the filtered hourly data if needed for other analyses
# hourly_output_filename = os.path.join(OUTPUT_DIR, "aqi_hourly_data_top6_most_data.csv")
# df_filtered_hourly.to_csv(hourly_output_filename, index=False)
# print(f"Filtered hourly data saved to: {hourly_output_filename}")

# --- 5. Aggregate Filtered Data to Daily Averages ---
print("\nAggregating filtered data to daily averages...")
df_filtered_hourly['date'] = df_filtered_hourly[DATE_COL].dt.date # Create date column for grouping

# Define aggregation dictionary dynamically based on available pollutant columns
agg_dict = {}
available_pollutants = [p for p in POLLUTANT_COLS if p in df_filtered_hourly.columns]
for pollutant in available_pollutants:
    agg_dict[pollutant] = 'mean' # Calculate mean for pollutants

# Define grouping columns
grouping_cols = ['date', STATE_COL, CITY_COL]

# *** FIX: No need to include STATE_COL or CITY_COL in agg_dict ***
# They are already grouping keys and will be preserved by reset_index()
# agg_dict[STATE_COL] = 'first' # REMOVE THIS LINE
# agg_dict[CITY_COL] = 'first' # REMOVE THIS LINE


# Perform aggregation
df_daily_agg = df_filtered_hourly.groupby(grouping_cols).agg(agg_dict).reset_index() # reset_index brings date, state, city back as columns

# Rename 'date' column to 'ds' for consistency with forecasting models
df_daily_agg.rename(columns={'date': 'ds'}, inplace=True)
df_daily_agg['ds'] = pd.to_datetime(df_daily_agg['ds']) # Ensure ds is datetime object

print(f"Shape after daily aggregation: {df_daily_agg.shape}")

# --- 6. Calculate Daily AQI ---
print("\nCalculating Daily AQI for aggregated data...")

# --- CPCB AQI Calculation Logic (Ensure functions are defined) ---
breakpoints = {
    'PM2_5': [ (0, 30, 0, 50), (31, 60, 51, 100), (61, 90, 101, 200), (91, 120, 201, 300), (121, 250, 301, 400), (251, float('inf'), 401, 500)],
    'PM10': [ (0, 50, 0, 50), (51, 100, 51, 100), (101, 250, 101, 200), (251, 350, 201, 300), (351, 430, 301, 400), (431, float('inf'), 401, 500)],
    'SO2': [ (0, 40, 0, 50), (41, 80, 51, 100), (81, 380, 101, 200), (381, 800, 201, 300), (801, 1600, 301, 400), (1601, float('inf'), 401, 500)],
    'NO2': [ (0, 40, 0, 50), (41, 80, 51, 100), (81, 180, 101, 200), (181, 280, 201, 300), (281, 400, 301, 400), (401, float('inf'), 401, 500)],
    'CO': [ (0, 1.0, 0, 50), (1.1, 2.0, 51, 100), (2.1, 10.0, 101, 200), (10.1, 17.0, 201, 300), (17.1, 34.0, 301, 400), (34.1, float('inf'), 401, 500)],
    'NH3': [ (0, 200, 0, 50), (201, 400, 51, 100), (401, 800, 101, 200), (801, 1200, 201, 300), (1201, 1800, 301, 400), (1801, float('inf'), 401, 500)]
    # Add Ozone if available and needed
}
pollutants_to_consider_aqi = [p for p in available_pollutants if p in breakpoints]

def calculate_sub_index_pred(pollutant, concentration):
    if pd.isna(concentration): return np.nan
    bp_list = breakpoints.get(pollutant)
    if not bp_list: return np.nan
    concentration = float(concentration)
    bp = None
    for conc_low, conc_high, aqi_low, aqi_high in bp_list:
        if conc_low == 0:
             if concentration >= conc_low and concentration <= conc_high: bp = (conc_low, conc_high, aqi_low, aqi_high); break
        else:
             if concentration > conc_low and concentration <= conc_high: bp = (conc_low, conc_high, aqi_low, aqi_high); break
    if bp is None and concentration > bp_list[-1][1]: bp = bp_list[-1]
    if bp is None: return np.nan
    conc_low, conc_high, aqi_low, aqi_high = bp
    if conc_high == conc_low: sub_index = aqi_low
    else: sub_index = ((aqi_high - aqi_low) / (conc_high - conc_low)) * (concentration - conc_low) + aqi_low
    return min(round(sub_index), 500)

def calculate_overall_aqi_pred(row):
    max_sub_index = 0; dominant_pollutant = 'N/A'; calculated_any = False
    available_pollutants_row = [p for p in pollutants_to_consider_aqi if p in row.index]
    for pollutant in available_pollutants_row:
        sub_index = calculate_sub_index_pred(pollutant, row[pollutant])
        if not pd.isna(sub_index):
            calculated_any = True
            if sub_index > max_sub_index: max_sub_index = sub_index; dominant_pollutant = pollutant
    if not calculated_any: return pd.Series([np.nan, 'N/A'], index=['AQI', 'Dominant_Pollutant'])
    else: return pd.Series([max_sub_index, dominant_pollutant], index=['AQI', 'Dominant_Pollutant'])
# --- End AQI Logic ---

# Apply AQI calculation row-wise
tqdm.pandas(desc="Calculating AQI")
aqi_results = df_daily_agg.progress_apply(calculate_overall_aqi_pred, axis=1)
df_daily_final = pd.concat([df_daily_agg, aqi_results], axis=1)

# Handle potential NaNs in AQI if all pollutants were missing for a day
df_daily_final['AQI'] = df_daily_final['AQI'].fillna(-1).astype(int) # Fill NaNs with -1 or another indicator
df_daily_final['Dominant_Pollutant'] = df_daily_final['Dominant_Pollutant'].fillna('N/A')

print("AQI calculation complete.")
print(df_daily_final[['ds', CITY_COL, 'AQI', 'Dominant_Pollutant']].head())


# --- 7. Save Final Daily Aggregated Data ---
daily_output_filename = os.path.join(OUTPUT_DIR, "aqi_daily_data_top6_most_data.csv")

# Ensure column order for the final CSV
final_cols = ['ds', STATE_COL, CITY_COL] + available_pollutants + ['AQI', 'Dominant_Pollutant']
df_daily_final = df_daily_final[final_cols] # Reorder/select columns

df_daily_final.to_csv(daily_output_filename, index=False, float_format='%.2f')
print(f"\nFinal daily aggregated data for top 6 cities saved to: {daily_output_filename}")
print(f"Final shape: {df_daily_final.shape}")
print("\nFirst 5 rows of final data:")
print(df_daily_final.head())

# --- Cleanup ---
del df_unified, df_filtered_hourly, df_daily_agg, aqi_results # Release memory
gc.collect()
print("\nProcessing complete.")


Configuration set.
Loading unified data from: /kaggle/input/6cityyyyy/unified_air_quality.csv
Loaded unified data shape: (14295663, 21)
Columns found: ['station_id', 'datetime', 'PM2_5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'SO2', 'CO', 'state', 'city', 'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'season', 'state_city', 'date']

Identifying top 6 cities based on data availability...

Top 6 cities selected: ['Delhi', 'Bengaluru', 'Mumbai', 'Chennai', 'Lucknow', 'Hyderabad']
Data counts for selected cities:
city
Delhi        2796171
Bengaluru     689460
Mumbai        638676
Chennai       472086
Lucknow       450305
Hyderabad     431872
Name: count, dtype: int64

Filtering unified data for 6 cities...
Shape after filtering (hourly): (5478570, 21)

Aggregating filtered data to daily averages...
Shape after daily aggregation: (28694, 10)

Calculating Daily AQI for aggregated data...


Calculating AQI:   0%|          | 0/28694 [00:00<?, ?it/s]

AQI calculation complete.
          ds       city  AQI Dominant_Pollutant
0 2010-01-01      Delhi  183              PM2_5
1 2010-01-01  Bengaluru   70               PM10
2 2010-01-01     Mumbai   87               PM10
3 2010-01-01    Chennai   55              PM2_5
4 2010-01-01    Lucknow  155              PM2_5

Final daily aggregated data for top 6 cities saved to: /kaggle/working/aqi_daily_data_top6_most_data.csv
Final shape: (28694, 12)

First 5 rows of final data:
          ds          state       city      PM2_5        PM10         NO  \
0 2010-01-01          Delhi      Delhi  84.916000  187.080000  13.479667   
1 2010-01-01      Karnataka  Bengaluru  28.180000   69.520000   4.014167   
2 2010-01-01    Maharashtra     Mumbai  26.610000   87.400000  17.160000   
3 2010-01-01     Tamil Nadu    Chennai  33.473333   53.363333  10.322361   
4 2010-01-01  Uttar Pradesh    Lucknow  76.840000  145.615000   8.894167   

         NO2        SO2        CO        NH3  AQI Dominant_Pollutant 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
